# Reproduce Figures – Ketkar & Sporar 2020

This notebook loads calcium imaging data from `.mat` files, processes them through the ported MATLAB pipeline, saves aligned trace + stimulus data to HDF5, and plots the results for visual verification against the published figures.

**Figures covered:**
- Figure 2B: 5s full-field flash (L2, L3)
- Figure 2C: 60s full-field flash (L3)
- Figure 2D-E: Contrast steps from grey baseline (L2, L3)
- Figure 3A: Adapting contrast steps (L2, L3)
- Figure 3F: Random luminance steps between 5 levels (L2, L3)

**Requirements:** `ketkar-data` conda environment with scipy, numpy, h5py, matplotlib, pandas, openpyxl.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import h5py
from pathlib import Path

import utils

%matplotlib inline
plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "font.size": 10,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

DATA_ROOT = Path("data/in vivo two-photon calcium imaging")
HDF5_PATH = Path("processed_data.h5")
summary = utils.load_summary()

In [ ]:
def plot_err_patch(ax, t, mean, sem, color, label=None):
    """Plot mean trace with shaded SEM, matching the paper style."""
    ax.fill_between(t, mean - sem, mean + sem, color=color, alpha=0.25)
    line, = ax.plot(t, mean, color=color, linewidth=1.5, label=label)
    return line

def process_and_save(h5f, group_name, rats, fly_ids, time_axis, stimulus,
                     cell_type, stimulus_type, roi_names=None,
                     epoch_labels=None, extra_attrs=None):
    """Run mean_cat_full and save everything to HDF5."""
    # Clean: remove all-zero / all-NaN ROIs
    if rats.ndim == 2:
        valid = (np.nansum(np.abs(rats), axis=1) > 0) & ~np.all(np.isnan(rats), axis=1)
    else:
        valid = (np.nansum(np.abs(rats), axis=tuple(range(1, rats.ndim))) > 0)

    rats_clean = rats[valid]
    fids_clean = fly_ids[valid]
    names_clean = [roi_names[i] for i in range(len(roi_names)) if valid[i]] if roi_names else None

    # Per-fly averaging
    if rats_clean.ndim == 2:
        fly_means, grand_mean, sem = utils.mean_cat_full(rats_clean, fids_clean)
    else:
        # For multi-epoch: reshape, average, reshape back
        orig_shape = rats_clean.shape
        flat = rats_clean.reshape(orig_shape[0], -1)
        fly_means_flat, grand_mean_flat, sem_flat = utils.mean_cat_full(flat, fids_clean)
        fly_means = fly_means_flat.reshape(fly_means_flat.shape[0], *orig_shape[1:])
        grand_mean = grand_mean_flat.reshape(orig_shape[1:])
        sem = sem_flat.reshape(orig_shape[1:])

    fly_ids_unique = np.unique(fids_clean)

    utils.save_figure_to_hdf5(
        h5f, group_name, rats_clean, fids_clean, time_axis, stimulus,
        cell_type, stimulus_type,
        fly_means=fly_means, grand_mean=grand_mean, sem=sem,
        fly_ids_unique=fly_ids_unique, roi_names=names_clean,
        epoch_labels=epoch_labels, extra_attrs=extra_attrs,
    )
    print(f"  Saved {group_name}: {rats_clean.shape[0]} ROIs, "
          f"{len(fly_ids_unique)} flies, shape={rats_clean.shape}")
    return rats_clean, fids_clean, fly_means, grand_mean, sem

## Figure 2B – 5s Full-Field Flash

ON/OFF stimulus with 5-second epochs. Shows transient calcium responses in L2 and L3 neurons.
dF/F normalized to whole-trace mean. Only negatively correlated ROIs are plotted (OFF-responding cells).

In [ ]:
h5f = h5py.File(HDF5_PATH, "a")

fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
colors = {"L2": "#3070B3", "L3": "#2E8B57"}

for ax, cell_type in zip(axes, ["L2", "L3"]):
    data_dir = DATA_ROOT / "Figure2" / "Figure2B" / f"{cell_type}_pData"
    print(f"Loading {cell_type}...")
    roi_data = utils.load_figure_data(data_dir, summary, region="AT")
    agg = utils.aggregate_fff(roi_data)

    # Select negatively correlated ROIs
    neg, pos = utils.classify_rois_by_correlation(agg["rats"], agg["stims"])
    neg_rats = agg["rats"][neg]
    neg_ids = agg["flyID"][neg]
    neg_names = [agg["name"][i] for i in range(len(agg["name"])) if neg[i]]

    dur = neg_rats.shape[1]
    t = np.arange(dur) / utils.IRATE
    stim_trace = np.nanmean(agg["stims"][neg], axis=0)

    rats_c, fids_c, fly_m, gm, se = process_and_save(
        h5f, f"fig2b_{cell_type}", neg_rats, neg_ids, t, stim_trace,
        cell_type, "5s_full_field_flash", roi_names=neg_names)

    plot_err_patch(ax, t, gm, se, colors[cell_type],
                   label=f"{cell_type} (n={len(np.unique(fids_c))} flies, {len(rats_c)} ROIs)")
    # Stimulus overlay
    ax.plot(t, stim_trace * 0.1 + np.max(gm + se) * 1.1, "k", linewidth=1)
    ax.axhline(0, color="gray", linewidth=0.5)
    ax.set_xlabel("Time (s)")
    ax.set_ylabel("dF/F")
    ax.set_title(f"Figure 2B – {cell_type} >> GCaMP6f")
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()
print("Done.")

## Figure 2C – 60s Full-Field Flash

Sustained response to a 60-second ON stimulus in L3 neurons. Shows that L3 maintains a tonic response throughout the stimulus.

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(10, 3))

data_dir = DATA_ROOT / "Figure2" / "Figure2C" / "L3_pData"
print("Loading L3 60s FFF...")
roi_data = utils.load_figure_data(data_dir, summary, region="AT")
agg = utils.aggregate_fff60s(roi_data)

neg, pos = utils.classify_rois_by_correlation(agg["rats"], agg["stims"])
neg_rats = agg["rats"][neg]
neg_ids = agg["flyID"][neg]
neg_names = [agg["name"][i] for i in range(len(agg["name"])) if neg[i]]

dur = neg_rats.shape[1]
t = np.arange(dur) / utils.IRATE
stim_trace = np.nanmean(agg["stims"][neg], axis=0)

rats_c, fids_c, fly_m, gm, se = process_and_save(
    h5f, "fig2c_L3", neg_rats, neg_ids, t, stim_trace,
    "L3", "60s_full_field_flash", roi_names=neg_names)

plot_err_patch(ax, t, gm, se, colors["L3"],
               label=f"L3 (n={len(np.unique(fids_c))} flies, {len(rats_c)} ROIs)")
ax.plot(t, stim_trace * 0.1 + np.max(gm + se) * 1.1, "k", linewidth=1)
ax.axhline(0, color="gray", linewidth=0.5)
ax.set_xlabel("Time (s)")
ax.set_ylabel("dF/F")
ax.set_title("Figure 2C – L3 >> GCaMP6f, 60s flash")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

## Figure 2D-E – Contrast Steps from Grey Baseline

10 contrast epochs (5 OFF: grey→dark at 100-20%, 5 ON: bright→grey at 50-16%).
dF/F normalized to grey baseline. Shows contrast-response relationship.

In [ ]:
contrast_labels = [
    "-100% OFF", "-80% OFF", "-60% OFF", "-40% OFF", "-20% OFF",
    "20% ON", "40% ON", "60% ON", "80% ON", "100% ON",
]

for cell_type in ["L2", "L3"]:
    data_dir = DATA_ROOT / "Figure2" / "Figure2D_G" / f"{cell_type}_pData"
    print(f"Loading {cell_type}...")
    roi_data = utils.load_figure_data(data_dir, summary, region="AT")
    agg = utils.aggregate_standing_stripe(roi_data)

    n_epoch = agg["n_epoch"]
    full_dur = agg["rats"].shape[2]
    t = np.arange(full_dur) / utils.IRATE

    rats_c, fids_c, fly_m, gm, se = process_and_save(
        h5f, f"fig2dg_{cell_type}", agg["rats"], agg["flyID"], t,
        agg["stimstruct"], cell_type, "contrast_steps_grey_baseline",
        roi_names=agg["name"], epoch_labels=contrast_labels,
        extra_attrs={"contrast_values": [-100, -80, -60, -40, -20, 20, 40, 60, 80, 100]})

    # Plot all 10 epochs
    fig, axes = plt.subplots(2, 5, figsize=(16, 6), sharey=True, sharex=True)
    for ep_idx, ax in enumerate(axes.flat):
        ep_mean = gm[ep_idx]
        ep_sem = se[ep_idx]
        plot_err_patch(ax, t, ep_mean, ep_sem, colors[cell_type])
        ax.plot(t, agg["stimstruct"] * 0.1 + np.max(ep_mean + ep_sem) * 1.1, "k", lw=0.8)
        ax.axhline(0, color="gray", linewidth=0.5)
        ax.set_title(contrast_labels[ep_idx], fontsize=8)
        if ep_idx % 5 == 0:
            ax.set_ylabel("dF/F")
        if ep_idx >= 5:
            ax.set_xlabel("Time (s)")
    fig.suptitle(f"Figure 2{'D' if cell_type == 'L2' else 'E'} – {cell_type} >> GCaMP6f", fontsize=12)
    plt.tight_layout()
    plt.show()

## Figure 3A – Adapting Contrast Steps

OFF→OFF stimulus at 6 different adapting backgrounds (luminances: 75, 64, 54, 43, 32, 21 cd/m²).
Shows how the B-step response depends on the adapting background luminance.
7 epoch combinations extracted via cantor pairing. Each epoch pair = 36s (360 timepoints at 10Hz).

In [ ]:
adapt_epoch_labels = [
    "100%OFF -> 50%OFF",
    "100%OFF -> GREY",
    "100%OFF -> 50%ON",
    "100%OFF -> 100%ON",
    "50%OFF -> 100%OFF",
    "50%OFF -> GREY",
    "50%OFF -> 50%ON",
]
luminance_values = [75, 64, 54, 43, 32, 21]

for cell_type in ["L2", "L3"]:
    data_dir = DATA_ROOT / "Figure3" / "Figure3A_E" / f"{cell_type}_pData"
    print(f"Loading {cell_type}...")
    roi_data = utils.load_figure_data(data_dir, summary)
    agg = utils.aggregate_adapting_contrast(roi_data, off_stimulus=True)

    # Classify ROIs
    neg_mask = utils.classify_rois_adapting(agg["rats"], agg["flyID"])
    neg_rats = agg["rats"][neg_mask]
    neg_ids = agg["flyID"][neg_mask]
    neg_names = [agg["name"][i] for i in range(len(agg["name"])) if neg_mask[i]]

    total_len = neg_rats.shape[2]  # 360
    t = np.arange(total_len) / utils.IRATE

    # Stimulus trace: just a step pattern (background for 300 samples, then step for 60)
    stim_pattern = np.zeros(total_len)
    stim_pattern[300:] = 1.0

    rats_c, fids_c, fly_m, gm, se = process_and_save(
        h5f, f"fig3a_{cell_type}", neg_rats, neg_ids, t, stim_pattern,
        cell_type, "adapting_contrast_OFF_OFF",
        roi_names=neg_names, epoch_labels=adapt_epoch_labels,
        extra_attrs={"luminance_values_cdm2": luminance_values})

    # Plot: overlay all epoch combinations (skip the baseline epoch)
    fig, ax = plt.subplots(figsize=(10, 4))
    epoch_colors = plt.cm.viridis(np.linspace(0, 0.9, 7))
    for ep_idx in range(min(6, gm.shape[0])):  # plot first 6 (skip 7th if partial)
        plot_err_patch(ax, t, gm[ep_idx] * 100, se[ep_idx] * 100,
                       epoch_colors[ep_idx], label=adapt_epoch_labels[ep_idx])
    ax.axhline(0, color="gray", linewidth=0.5)
    ax.set_xlabel("Time (s)")
    ax.set_ylabel("dF/F (%)")
    ax.set_title(f"Figure 3A – {cell_type} >> GCaMP6f, adapting contrast")
    ax.legend(fontsize=7, loc="upper right")
    plt.tight_layout()
    plt.show()

## Figure 3F – Random Luminance Steps (5 Levels)

Stimulus randomly switches between 5 luminance levels (100%OFF, 50%OFF, GREY, 50%ON, 100%ON), each held ~10s.
20 unique transition types are extracted. dF/F normalized to 100%ON baseline.
Shows responses grouped by transition type.

In [ ]:
for cell_type in ["L2", "L3"]:
    data_dir = DATA_ROOT / "Figure3" / "Figure3F_H_I" / f"{cell_type}_pData"
    print(f"Loading {cell_type}...")
    roi_data = utils.load_figure_data(data_dir, summary)
    agg = utils.aggregate_05steps(roi_data)

    # Classify ROIs
    neg_mask, pos_mask = utils.classify_rois_05steps(agg["rats"])
    neg_rats = agg["rats"][neg_mask]
    neg_ids = agg["flyID"][neg_mask]
    neg_names = [agg["name"][i] for i in range(len(agg["name"])) if neg_mask[i]]

    n_time = neg_rats.shape[2]
    t = np.arange(n_time) / utils.IRATE

    # Stimulus pattern: first epoch then second epoch
    stim_pattern = np.zeros(n_time)
    stim_pattern[n_time // 2:] = 1.0

    rats_c, fids_c, fly_m, gm, se = process_and_save(
        h5f, f"fig3f_{cell_type}", neg_rats, neg_ids, t, stim_pattern,
        cell_type, "random_5level_luminance_steps",
        roi_names=neg_names, epoch_labels=agg["transition_labels"],
        extra_attrs={"epochlength_samples": agg["epochlength"]})

    # Plot: 4x5 grid of transitions
    fig, axes = plt.subplots(4, 5, figsize=(18, 12), sharey=True, sharex=True)
    trans_colors = plt.cm.tab20(np.linspace(0, 1, 20))
    for ep_idx, ax in enumerate(axes.flat):
        ep_mean = gm[ep_idx]
        ep_sem = se[ep_idx]
        plot_err_patch(ax, t, ep_mean, ep_sem, trans_colors[ep_idx])
        ax.axhline(0, color="gray", linewidth=0.5)
        ax.axvline(t[n_time // 2], color="gray", linewidth=0.5, linestyle="--")
        ax.set_title(agg["transition_labels"][ep_idx], fontsize=7)
        if ep_idx % 5 == 0:
            ax.set_ylabel("dF/F")
        if ep_idx >= 15:
            ax.set_xlabel("Time (s)")
    fig.suptitle(f"Figure 3F – {cell_type} >> GCaMP6f, random steps", fontsize=12)
    plt.tight_layout()
    plt.show()

In [ ]:
h5f.close()
print(f"\nAll data saved to {HDF5_PATH}")
print(f"File size: {HDF5_PATH.stat().st_size / 1e6:.1f} MB")

# List contents
with h5py.File(HDF5_PATH, "r") as f:
    print("\nHDF5 contents:")
    for key in f.keys():
        grp = f[key]
        attrs = dict(grp.attrs)
        print(f"  {key}: {attrs.get('cell_type', '?')}, "
              f"{attrs.get('stimulus_type', '?')}, "
              f"n_rois={attrs.get('n_rois', '?')}, "
              f"n_flies={attrs.get('n_flies', '?')}")